# Exploratory Data Analysis: Synthetic Clinical Requisitions

This notebook explores the generated synthetic requisition dataset. It uses only synthetic data and does not train a model.

## 1. Load The Dataset

The generated CSV should already exist at `data/raw/synthetic_clinical_requisitions.csv` after running `src/data/generate_synthetic_data.py`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
plt.style.use("default")

In [ ]:
candidate_paths = [
    Path("../data/raw/synthetic_clinical_requisitions.csv"),
    Path("data/raw/synthetic_clinical_requisitions.csv"),
]

data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Run src/data/generate_synthetic_data.py before this notebook.")

df = pd.read_csv(data_path)
print(f"Loaded dataset from: {data_path}")

**Observation:** The notebook starts from the generated CSV, not from real requisition forms. This keeps the analysis reproducible and portfolio-safe.

## 2. Basic Structure

This section checks the dataset shape, column names, data types, and a few sample records.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

display(pd.DataFrame({"columns": df.columns}))
display(df.dtypes.to_frame("dtype"))
display(df.head(10))

**Observation:** Each row represents one synthetic requisition. The boolean columns describe whether important fields are present, while `requires_manual_review` is the label we will eventually predict in a later milestone.

## 3. Missing Values And Duplicates

A clean synthetic dataset should not contain blank values, and each `document_id` should be unique.

In [ ]:
missing_values = df.isna().sum().to_frame("missing_values")
duplicate_rows = int(df.duplicated().sum())
duplicate_document_ids = int(df["document_id"].duplicated().sum())

display(missing_values)
print(f"Duplicate full rows: {duplicate_rows}")
print(f"Duplicate document IDs: {duplicate_document_ids}")

**Observation:** Missing values and duplicate document IDs would weaken the dataset because the model could learn from incomplete or repeated examples. For this synthetic milestone, we expect no missing cell values and no duplicate `document_id` values.

## 4. Class Distribution

The class distribution shows how many records require manual review and how many do not.

In [ ]:
class_counts = df["requires_manual_review"].value_counts().reindex([False, True], fill_value=0)
class_percent = df["requires_manual_review"].value_counts(normalize=True).reindex([False, True], fill_value=0).mul(100)
class_summary = pd.DataFrame({
    "label": ["No manual review", "Requires manual review"],
    "count": class_counts.values,
    "percent": class_percent.round(2).values,
})

display(class_summary)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(class_summary["label"], class_summary["count"], color=["#4C78A8", "#F58518"])
ax.set_title("Manual Review Class Distribution")
ax.set_ylabel("Record Count")
ax.set_xlabel("Class")
for index, value in enumerate(class_summary["count"]):
    ax.text(index, value, f"{value:,}", ha="center", va="bottom")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

**Observation:** The manual-review class is expected to be smaller because only records that trip one of the synthetic rules are labeled for review. This matters later because a model could become too comfortable predicting the majority class.

## 5. Manual-Review Rate

The manual-review rate is the percentage of records where `requires_manual_review` is `True`.

In [ ]:
manual_review_rate = df["requires_manual_review"].mean()
print(f"Manual-review rate: {manual_review_rate:.2%}")

**Observation:** This single number gives a quick sense of how often the synthetic workflow sends records to manual review. In a future modeling step, this rate will help decide which evaluation metrics are more useful than accuracy alone.

## 6. Review Rate By Test Type

This checks whether some synthetic test categories have higher manual-review rates than others.

In [ ]:
review_by_test_type = (
    df.groupby("test_type")["requires_manual_review"]
    .agg(record_count="count", review_rate="mean")
    .sort_values("review_rate", ascending=False)
)
review_by_test_type["review_rate_percent"] = (review_by_test_type["review_rate"] * 100).round(2)
display(review_by_test_type)

plot_data = review_by_test_type.sort_values("review_rate_percent")
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(plot_data.index, plot_data["review_rate_percent"], color="#54A24B")
ax.set_title("Manual-Review Rate By Test Type")
ax.set_xlabel("Manual-Review Rate (%)")
ax.set_ylabel("Test Type")
plt.tight_layout()
plt.show()

**Observation:** The synthetic label rules do not directly use `test_type`, so very large differences by test type would be a signal to inspect the data-generation probabilities and random variation.

## 7. Review Rate By Specimen Type

This checks whether some synthetic specimen categories have higher manual-review rates.

In [ ]:
review_by_specimen_type = (
    df.groupby("specimen_type")["requires_manual_review"]
    .agg(record_count="count", review_rate="mean")
    .sort_values("review_rate", ascending=False)
)
review_by_specimen_type["review_rate_percent"] = (review_by_specimen_type["review_rate"] * 100).round(2)
display(review_by_specimen_type)

plot_data = review_by_specimen_type.sort_values("review_rate_percent")
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(plot_data.index, plot_data["review_rate_percent"], color="#E45756")
ax.set_title("Manual-Review Rate By Specimen Type")
ax.set_xlabel("Manual-Review Rate (%)")
ax.set_ylabel("Specimen Type")
plt.tight_layout()
plt.show()

**Observation:** Specimen type is useful for analysis, but it is not one of the current manual-review rules. Differences here should be treated as exploratory signals, not real clinical conclusions.

## 8. OCR Confidence Analysis

`ocr_confidence` is simulated. It does not come from real OCR. The synthetic rule sends records to manual review when OCR confidence is below `0.82`.

In [ ]:
display(df["ocr_confidence"].describe().to_frame("ocr_confidence"))
display(df.groupby("requires_manual_review")["ocr_confidence"].describe())

fig, ax = plt.subplots(figsize=(8, 4.5))
for label, group in df.groupby("requires_manual_review"):
    chart_label = "Requires manual review" if label else "No manual review"
    ax.hist(group["ocr_confidence"], bins=20, alpha=0.65, label=chart_label)
ax.axvline(0.82, color="black", linestyle="--", linewidth=1.5, label="Review threshold: 0.82")
ax.set_title("OCR Confidence Distribution By Review Label")
ax.set_xlabel("OCR Confidence")
ax.set_ylabel("Record Count")
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.boxplot(
    [
        df.loc[~df["requires_manual_review"], "ocr_confidence"],
        df.loc[df["requires_manual_review"], "ocr_confidence"],
    ],
)
ax.set_xticks([1, 2], ["No Review", "Manual Review"])
ax.axhline(0.82, color="black", linestyle="--", linewidth=1.2)
ax.set_title("OCR Confidence By Review Label")
ax.set_ylabel("OCR Confidence")
plt.tight_layout()
plt.show()

**Observation:** OCR confidence should be lower among many manual-review records because low confidence is one of the synthetic review triggers. Some reviewed records can still have high OCR confidence if they were reviewed for another reason, such as duplicate kit or missing fields.

## 9. Possible Class Imbalance

This section checks whether one class is much smaller than the other.

In [ ]:
positive_rate = df["requires_manual_review"].mean()
negative_rate = 1 - positive_rate
minority_rate = min(positive_rate, negative_rate)

print(f"No-review rate: {negative_rate:.2%}")
print(f"Manual-review rate: {positive_rate:.2%}")
print(f"Minority-class rate: {minority_rate:.2%}")

if minority_rate < 0.20:
    print("Possible class imbalance detected: one class is below 20% of the dataset.")
else:
    print("No strong class imbalance detected using the 20% rule of thumb.")

**Observation:** Class imbalance is not automatically bad, but it changes how future models should be evaluated. In a later milestone, metrics like recall, precision, F1 score, and a stratified train/test split will matter more than accuracy alone.

## 10. EDA Summary

This notebook confirms that the dataset can be loaded, inspected, and analyzed with plain pandas and matplotlib. The next milestone can use these findings to design preprocessing and model-evaluation steps, but no model is trained here.